In [1]:
# ============================================================
# PS S6E5 - exp_20260503_008_realmlp_shared001_plus_006b_orig_prior
# RealMLP shared001 min + safe original prior graft
#
# Base:
# - 007 RealMLP shared001 minimum reproduction
#
# Add:
# - ORIG_PRIOR_* mean features from original dataset
# - ORIG_PRIOR_*_minus_global
#
# Exclude:
# - Normalized_TyreLife
# - Normalized_TyreLife proxy
# - ORIG_PRIOR_Year
# - all *_count features
# - Race x Driver x Stint fine prior
# ============================================================

In [2]:
# If pytabkit is not installed in the notebook environment, uncomment:
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.5 MB/s eta 0:00:00


In [3]:
import os
import gc
import json
import random
import warnings
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260503_008_realmlp_shared001_plus_006b_orig_prior"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 42
    N_SPLITS = 5
    USE_TE = True
    USE_ORIGINAL_ROWS = True

    # shared_001 params
    PARAMS = {
        "random_state": 42,
        "verbosity": 2,
        "val_metric_name": "1-auc_ovr",

        "n_ens": 8,
        "n_epochs": 4,
        "batch_size": 256,
        "use_early_stopping": False,
        "early_stopping_additive_patience": 10,
        "early_stopping_multiplicative_patience": 1,

        "lr": 0.03,
        "wd": 0.018,
        "sq_mom": 0.98,
        "lr_sched": "lin_cos_log_15",
        "first_layer_lr_factor": 0.25,

        "embedding_size": 6,
        "max_one_hot_cat_size": 18,
        "hidden_sizes": [512, 256, 128],
        "act": "silu",
        "p_drop": 0.05,
        "p_drop_sched": "expm4t",

        "plr_hidden_1": 16,
        "plr_hidden_2": 8,
        "plr_act_name": "gelu",
        "plr_lr_factor": 0.1151,
        "plr_sigma": 2.33,

        "ls_eps": 0.01,
        "ls_eps_sched": "sqrt_cos",

        "add_front_scale": False,
        "bias_init_mode": "neg-uniform-dynamic-2",
        "tfms": [
            "one_hot",
            "median_center",
            "robust_scale",
            "smooth_clip",
            "embedding",
            "l2_normalize",
        ],
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [5]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_realmlp_runtime_params(params: dict) -> dict:
    """
    P100では現在のPyTorch/CUDA wheelとpytabkitが合わず、
    CUDA error: no kernel image is available for execution on the device
    が出ることがあるため、P100のみCPU fallbackにする。
    T4/L4/A100ではCUDAを使う。
    """
    params = params.copy()

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
    else:
        gpu_name = "CPU"

    is_p100 = "P100" in gpu_name.upper()

    if is_p100:
        print(f"[Runtime] GPU is {gpu_name}. Forcing RealMLP to CPU due to CUDA kernel compatibility.")
        params["device"] = "cpu"
        params["n_ens"] = 2
        params["batch_size"] = 512
        params["verbosity"] = 1

    elif torch.cuda.is_available():
        print(f"[Runtime] GPU is {gpu_name}. Using CUDA for RealMLP.")
        params["device"] = "cuda"

    else:
        print("[Runtime] CUDA is not available. Using CPU for RealMLP.")
        params["device"] = "cpu"
        params["n_ens"] = 2
        params["batch_size"] = 512
        params["verbosity"] = 1

    return params


seed_everything(CFG.SEED)

REALMLP_PARAMS = configure_realmlp_runtime_params(CFG.PARAMS)

print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device     :", torch.cuda.get_device_name(0))
print("RealMLP device  :", REALMLP_PARAMS.get("device"))
print("RealMLP n_ens   :", REALMLP_PARAMS.get("n_ens"))
print("RealMLP batch   :", REALMLP_PARAMS.get("batch_size"))

[Runtime] GPU is Tesla T4. Using CUDA for RealMLP.
PyTorch  version: 2.10.0+cu128
PyTabKit version: 1.7.3
CUDA available  : True
CUDA device     : Tesla T4
RealMLP device  : cuda
RealMLP n_ens   : 8
RealMLP batch   : 256


In [6]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)
orig_full = pd.read_csv(CFG.ORIG_PATH)

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sub shape  :", sub.shape)
print("orig shape :", orig_full.shape)

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns
assert CFG.TARGET in orig_full.columns
assert CFG.DANGER_COL in orig_full.columns

print("\ncompetition target mean:", train[CFG.TARGET].mean())
print("original target mean   :", orig_full[CFG.TARGET].mean())

train shape: (439140, 16)
test shape : (188165, 15)
sub shape  : (188165, 2)
orig shape : (101371, 16)

competition target mean: 0.19898210137996994
original target mean   : 0.25479673673930414


In [7]:
# ============================================================
# Base preprocessing
# ============================================================

# Drop danger column immediately
orig_full = orig_full.drop(columns=[CFG.DANGER_COL])

y_orig = orig_full[CFG.TARGET].astype(int).copy()
orig = orig_full.drop(columns=[CFG.TARGET]).copy()

X = train.drop(columns=[CFG.ID_COL, CFG.TARGET]).copy()
y = train[CFG.TARGET].astype(int).copy()
train_id = train[CFG.ID_COL].copy()

X_test = test.drop(columns=[CFG.ID_COL]).copy()
test_id = test[CFG.ID_COL].copy()

print("\nX init shape     :", X.shape)
print("X_test init shape:", X_test.shape)
print("orig init shape  :", orig.shape)

base_cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
base_num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("\nbase cat cols:", len(base_cat_cols), base_cat_cols)
print("base num cols:", len(base_num_cols), base_num_cols)


X init shape     : (439140, 14)
X_test init shape: (188165, 14)
orig init shape  : (101371, 14)

base cat cols: 3 ['Driver', 'Compound', 'Race']
base num cols: 11 ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']


In [8]:
# ============================================================
# shared_001 Feature Engineering
# ============================================================

category_map = {}

important_combos = [
    ("Race", "Compound"),
    ("Race", "Year"),
]

def feature_engineering(df: pd.DataFrame, fit: bool = False):
    df = df.copy()

    # Categorize numerical columns by floor value.
    # Use only original shared_001 numerical columns here.
    for col in base_num_cols:
        cat_name = f"{col}_cat_"

        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype("int32")

        df[cat_name] = codes.astype(str)

    # Arithmetic interactions
    df["_LapNumber_/_RaceProgress"] = (
        df["LapNumber"] / (df["RaceProgress"] + 1e-6)
    ).astype("float32")

    df["_TyreLife_/_LapNumber"] = (
        df["TyreLife"] / df["LapNumber"].clip(lower=1)
    ).astype("float32")

    # RaceProgress quantile bin
    bin_config = {
        "RaceProgress": [200],
    }

    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ["quantile"]:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"

                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode="ordinal",
                        strategy=strategy,
                        subsample=None,
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype("int32")

                df[bin_name] = binned.astype(str)

    # Interaction categories
    combo_names = []

    for col1, col2 in important_combos:
        combo_name = f"{col1}_{col2}_"
        combo_names.append(combo_name)

        combo_series = df[col1].astype(str) + "_" + df[col2].astype(str)

        if fit:
            codes, uniques = combo_series.factorize()
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype("int32")

        df[combo_name] = codes.astype(str)

    new_cat_cols = [col for col in df.columns if col.endswith("_")]
    new_num_cols = [col for col in df.columns if col.startswith("_")]

    return df, new_cat_cols, new_num_cols, combo_names


X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)

print("\nAfter shared_001 FE")
print("X shape     :", X.shape)
print("X_test shape:", X_test.shape)
print("orig shape  :", orig.shape)
print("new_cat_cols:", len(new_cat_cols))
print("new_num_cols:", len(new_num_cols))
print("combo_names :", combo_names)


After shared_001 FE
X shape     : (439140, 30)
X_test shape: (188165, 30)
orig shape  : (101371, 30)
new_cat_cols: 14
new_num_cols: 2
combo_names : ['Race_Compound_', 'Race_Year_']


In [9]:
# ============================================================
# 008 Add safe original prior features
# ============================================================

def add_safe_original_prior_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
    y_orig: pd.Series,
    target_col: str,
):
    """
    Add safe original prior features.

    Policy:
    - original target is used only to create coarse prior means.
    - no Normalized_TyreLife.
    - no Year prior.
    - no count features.
    - no Race x Driver x Stint fine prior.
    - original rows are not used as validation.
    """

    train_df = train_df.copy()
    test_df = test_df.copy()
    orig_df = orig_df.copy()

    orig_with_target = orig_df.copy()
    orig_with_target[target_col] = y_orig.values

    global_prior = float(orig_with_target[target_col].mean())

    added = []
    prior_info = {
        "global_prior": global_prior,
        "year_prior_used": False,
        "count_features_used": False,
        "categorical_prior_cols": [],
        "numeric_bin_prior_cols": [],
        "pair_prior_cols": [],
        "added_features": [],
        "added_feature_count": 0,
    }

    # ----------------------------
    # 1. Categorical original priors
    # ----------------------------
    cat_prior_cols = [
        "Driver",
        "Compound",
        "Race",
        # "Year",  # excluded
        "Stint",
        "PitStop",
        "Position",
    ]

    for col in cat_prior_cols:
        if col not in orig_with_target.columns:
            continue
        if col not in train_df.columns or col not in test_df.columns:
            continue

        prior_name = f"ORIG_PRIOR_{col}"

        prior = (
            orig_with_target
            .groupby(col, dropna=False)[target_col]
            .agg("mean")
            .reset_index()
            .rename(columns={target_col: prior_name})
        )

        train_df = train_df.merge(prior, on=col, how="left")
        test_df = test_df.merge(prior, on=col, how="left")
        orig_df = orig_df.merge(prior, on=col, how="left")

        train_df[prior_name] = train_df[prior_name].fillna(global_prior).astype("float32")
        test_df[prior_name] = test_df[prior_name].fillna(global_prior).astype("float32")
        orig_df[prior_name] = orig_df[prior_name].fillna(global_prior).astype("float32")

        added.append(prior_name)
        prior_info["categorical_prior_cols"].append(col)

    # ----------------------------
    # 2. Numeric bin original priors
    # ----------------------------
    numeric_bin_cols = [
        "LapNumber",
        "TyreLife",
        "RaceProgress",
        "Cumulative_Degradation",
        "LapTime_Delta",
    ]

    numeric_bin_cols = [
        c for c in numeric_bin_cols
        if c in train_df.columns and c in test_df.columns and c in orig_with_target.columns
    ]

    bin_edges_info = {}

    for col in numeric_bin_cols:
        combined_values = pd.concat(
            [train_df[col], test_df[col], orig_with_target[col]],
            axis=0,
            ignore_index=True,
        ).dropna()

        if combined_values.nunique() <= 2:
            continue

        try:
            edges = np.unique(combined_values.quantile(np.linspace(0, 1, 11)).values)

            if len(edges) <= 2:
                continue

            edges[0] = -np.inf
            edges[-1] = np.inf

            bin_col = f"{col}_bin10_for_orig_prior"
            prior_name = f"ORIG_PRIOR_{col}_bin10"

            train_df[bin_col] = pd.cut(train_df[col], bins=edges, include_lowest=True)
            test_df[bin_col] = pd.cut(test_df[col], bins=edges, include_lowest=True)
            orig_df[bin_col] = pd.cut(orig_df[col], bins=edges, include_lowest=True)
            orig_with_target[bin_col] = pd.cut(orig_with_target[col], bins=edges, include_lowest=True)

            prior = (
                orig_with_target
                .groupby(bin_col, observed=True)[target_col]
                .agg("mean")
                .reset_index()
                .rename(columns={target_col: prior_name})
            )

            train_df = train_df.merge(prior, on=bin_col, how="left")
            test_df = test_df.merge(prior, on=bin_col, how="left")
            orig_df = orig_df.merge(prior, on=bin_col, how="left")

            train_df[prior_name] = train_df[prior_name].fillna(global_prior).astype("float32")
            test_df[prior_name] = test_df[prior_name].fillna(global_prior).astype("float32")
            orig_df[prior_name] = orig_df[prior_name].fillna(global_prior).astype("float32")

            train_df = train_df.drop(columns=[bin_col])
            test_df = test_df.drop(columns=[bin_col])
            orig_df = orig_df.drop(columns=[bin_col])
            orig_with_target = orig_with_target.drop(columns=[bin_col])

            added.append(prior_name)
            prior_info["numeric_bin_prior_cols"].append(col)
            bin_edges_info[col] = [
                float(x) if np.isfinite(x) else str(x)
                for x in edges
            ]

        except Exception as e:
            print(f"failed numeric bin prior for {col}:", e)

    prior_info["bin_edges"] = bin_edges_info

    # ----------------------------
    # 3. Coarse pair priors
    # ----------------------------
    pair_cols = [
        ("Compound", "Stint"),
        ("Compound", "PitStop"),
        ("Race", "Stint"),
        ("Race", "PitStop"),
    ]

    for c1, c2 in pair_cols:
        if c1 not in orig_with_target.columns or c2 not in orig_with_target.columns:
            continue
        if c1 not in train_df.columns or c2 not in train_df.columns:
            continue

        pair_key = f"{c1}__{c2}__orig_prior_key"
        prior_name = f"ORIG_PRIOR_{c1}__{c2}"

        for df in [train_df, test_df, orig_df, orig_with_target]:
            df[pair_key] = df[c1].astype(str) + "__" + df[c2].astype(str)

        prior = (
            orig_with_target
            .groupby(pair_key, dropna=False)[target_col]
            .agg("mean")
            .reset_index()
            .rename(columns={target_col: prior_name})
        )

        train_df = train_df.merge(prior, on=pair_key, how="left")
        test_df = test_df.merge(prior, on=pair_key, how="left")
        orig_df = orig_df.merge(prior, on=pair_key, how="left")

        train_df[prior_name] = train_df[prior_name].fillna(global_prior).astype("float32")
        test_df[prior_name] = test_df[prior_name].fillna(global_prior).astype("float32")
        orig_df[prior_name] = orig_df[prior_name].fillna(global_prior).astype("float32")

        train_df = train_df.drop(columns=[pair_key])
        test_df = test_df.drop(columns=[pair_key])
        orig_df = orig_df.drop(columns=[pair_key])
        orig_with_target = orig_with_target.drop(columns=[pair_key])

        added.append(prior_name)
        prior_info["pair_prior_cols"].append([c1, c2])

    # ----------------------------
    # 4. minus_global
    # ----------------------------
    prior_mean_cols = [
        c for c in added
        if c.startswith("ORIG_PRIOR_") and not c.endswith("_count")
    ]

    for c in prior_mean_cols:
        new_col = f"{c}_minus_global"

        train_df[new_col] = (train_df[c] - global_prior).astype("float32")
        test_df[new_col] = (test_df[c] - global_prior).astype("float32")
        orig_df[new_col] = (orig_df[c] - global_prior).astype("float32")

        added.append(new_col)

    # Safety checks
    forbidden_cols = [
        "ORIG_PRIOR_Year",
        "ORIG_PRIOR_Year_count",
        "ORIG_PRIOR_Year_minus_global",
    ]

    for c in forbidden_cols:
        assert c not in train_df.columns, f"{c} should not exist in train_df"
        assert c not in test_df.columns, f"{c} should not exist in test_df"
        assert c not in orig_df.columns, f"{c} should not exist in orig_df"

    count_cols = [c for c in train_df.columns if c.startswith("ORIG_PRIOR_") and c.endswith("_count")]
    assert len(count_cols) == 0, f"count features should not be created: {count_cols}"

    prior_info["added_features"] = added
    prior_info["added_feature_count"] = len(added)

    return train_df, test_df, orig_df, added, prior_info


X, X_test, orig, added_prior_features, prior_info = add_safe_original_prior_features(
    train_df=X,
    test_df=X_test,
    orig_df=orig,
    y_orig=y_orig,
    target_col=CFG.TARGET,
)

print("\nAfter adding safe original prior")
print("X shape     :", X.shape)
print("X_test shape:", X_test.shape)
print("orig shape  :", orig.shape)
print("added prior features:", len(added_prior_features))
for c in added_prior_features:
    print(c)

print("\nprior_info:")
print(json.dumps(prior_info, ensure_ascii=False, indent=2)[:5000])


After adding safe original prior
X shape     : (439140, 60)
X_test shape: (188165, 60)
orig shape  : (101371, 60)
added prior features: 30
ORIG_PRIOR_Driver
ORIG_PRIOR_Compound
ORIG_PRIOR_Race
ORIG_PRIOR_Stint
ORIG_PRIOR_PitStop
ORIG_PRIOR_Position
ORIG_PRIOR_LapNumber_bin10
ORIG_PRIOR_TyreLife_bin10
ORIG_PRIOR_RaceProgress_bin10
ORIG_PRIOR_Cumulative_Degradation_bin10
ORIG_PRIOR_LapTime_Delta_bin10
ORIG_PRIOR_Compound__Stint
ORIG_PRIOR_Compound__PitStop
ORIG_PRIOR_Race__Stint
ORIG_PRIOR_Race__PitStop
ORIG_PRIOR_Driver_minus_global
ORIG_PRIOR_Compound_minus_global
ORIG_PRIOR_Race_minus_global
ORIG_PRIOR_Stint_minus_global
ORIG_PRIOR_PitStop_minus_global
ORIG_PRIOR_Position_minus_global
ORIG_PRIOR_LapNumber_bin10_minus_global
ORIG_PRIOR_TyreLife_bin10_minus_global
ORIG_PRIOR_RaceProgress_bin10_minus_global
ORIG_PRIOR_Cumulative_Degradation_bin10_minus_global
ORIG_PRIOR_LapTime_Delta_bin10_minus_global
ORIG_PRIOR_Compound__Stint_minus_global
ORIG_PRIOR_Compound__PitStop_minus_global
ORI

In [10]:
# ============================================================
# Final feature metadata
# ============================================================

feature_cols = X.columns.tolist()

te_names = [f"_{col}TE" for col in combo_names]

print("\nfeature count before TE:", len(feature_cols))
print("target encoding columns:", combo_names)
print("target encoding generated:", te_names)


feature count before TE: 60
target encoding columns: ['Race_Compound_', 'Race_Year_']
target encoding generated: ['_Race_Compound_TE', '_Race_Year_TE']


In [11]:
# ============================================================
# CV Training
# ============================================================

skf_comp = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

skf_orig = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof = np.zeros(len(X), dtype=np.float32)
pred = np.zeros(len(X_test), dtype=np.float32)

fold_scores = []
feature_cols_by_fold = []

comp_splits = list(skf_comp.split(X, y))
orig_splits = list(skf_orig.split(orig, y_orig))

for fold, ((tr_idx, va_idx), (or_tr_idx, or_va_idx)) in enumerate(
    zip(comp_splits, orig_splits),
    1,
):
    print("\n" + "=" * 80)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 80)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy().reset_index(drop=True)

    if CFG.USE_ORIGINAL_ROWS:
        orig_tr = orig.iloc[or_tr_idx].copy()
        y_orig_tr = y_orig.iloc[or_tr_idx].copy().reset_index(drop=True)

        X_tr = pd.concat(
            [X_tr.reset_index(drop=True), orig_tr.reset_index(drop=True)],
            axis=0,
        )
        y_tr = pd.concat([y_tr, y_orig_tr], axis=0).reset_index(drop=True)

    X_va = X.iloc[va_idx].copy()
    y_va = y.iloc[va_idx].copy()

    X_te = X_test.copy()

    # Target encoding on shared_001 combo columns only.
    if CFG.USE_TE:
        te_cols = combo_names

        te = TargetEncoder(
            cv=CFG.N_SPLITS,
            smooth="auto",
            shuffle=True,
            random_state=CFG.SEED,
        )

        tr_enc = te.fit_transform(X_tr[te_cols], y_tr)
        va_enc = te.transform(X_va[te_cols])
        te_enc = te.transform(X_te[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]

        X_tr[te_names] = tr_enc
        X_va[te_names] = va_enc
        X_te[te_names] = te_enc

    if fold == 1:
        print("len(FEATURES):", len(X_tr.columns.tolist()))
        print("features:", X_tr.columns.tolist())

    print("train fold shape:", X_tr.shape)
    print("valid fold shape:", X_va.shape)
    print("test  fold shape:", X_te.shape)
    print("train target mean:", float(np.mean(y_tr)))
    print("valid target mean:", float(np.mean(y_va)))

    model = RealMLP_TD_Classifier(**REALMLP_PARAMS)
    model.fit(X_tr, y_tr, X_va, y_va)

    va_pred = model.predict_proba(X_va)[:, 1].astype(np.float32)
    te_pred = model.predict_proba(X_te)[:, 1].astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, va_pred)
    fold_scores.append(float(fold_auc))

    print(f"Fold {fold} AUC: {fold_auc:.8f}")

    feature_cols_by_fold.append(X_tr.columns.tolist())

    del model, X_tr, X_va, X_te
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


cv_auc = roc_auc_score(y, oof)

print("\n" + "=" * 80)
print("CV RESULT")
print("=" * 80)
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("fold mean:", float(np.mean(fold_scores)))
print("fold std :", float(np.std(fold_scores)))


Fold 1/5
len(FEATURES): 62
features: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', 'RaceProgress_200_quantile_bin_', 'Race_Compound_', 'Race_Year_', 'ORIG_PRIOR_Driver', 'ORIG_PRIOR_Compound', 'ORIG_PRIOR_Race', 'ORIG_PRIOR_Stint', 'ORIG_PRIOR_PitStop', 'ORIG_PRIOR_Position', 'ORIG_PRIOR_LapNumber_bin10', 'ORIG_PRIOR_TyreLife_bin10', 'ORIG_PRIOR_RaceProgress_bin10', 'ORIG_PRIOR_Cumulative_Degradation_bin10', 'ORIG_PRIOR_LapTime_Delta_bin10', 'ORIG_PRIOR_Compound__Stint', 'ORIG_PRIOR_Compound__PitStop', 'ORIG_PRIOR_Race__Stint', 'ORIG_PRIOR_Race__PitStop', 'ORIG_PRIOR_Driver_minus_gl

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/4: val 1-auc_ovr = 0.053540
Epoch 2/4: val 1-auc_ovr = 0.047874
Epoch 3/4: val 1-auc_ovr = 0.047327
Epoch 4/4: val 1-auc_ovr = 0.046059


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 AUC: 0.95394059

Fold 2/5
train fold shape: (432409, 62)
valid fold shape: (87828, 62)
test  fold shape: (188165, 62)
train target mean: 0.20945216218903862
valid target mean: 0.19897982420184906
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', 'ORIG_PRIOR_Driver', 'ORIG_PRIOR_Compound', 'ORIG_PRIOR_Race', 'ORIG_PRIOR_Stint', 'ORIG_PRIOR_PitStop', 'ORIG_PRIOR_Position', 'ORIG_PRIOR_LapNumber_bin10', 'ORIG_PRIOR_TyreLife_bin10', 'ORIG_PRIOR_RaceProgress_bin10', 'ORIG_PRIOR_Cumulative_Degradation_bin10', 'ORIG_PRIOR_LapTime_Delta_bin10', 'ORIG_PRIOR_Compound__Stint', 'ORIG_PRIOR_Compound__PitStop', 'ORIG_PRIOR_Race__Stint', 'ORIG_PRIOR_Race__PitStop', 'ORIG_PRIOR_Driver_minus_global', 'ORIG_PRIOR_Compound_minus_global', 'ORIG_PRIOR_Race_minus_global', 'ORIG_PRIOR_Stint_minus_global', 'O

...

Fold 4 AUC: 0.95228144

Fold 5/5
train fold shape: (432409, 62)
valid fold shape: (87828, 62)
test  fold shape: (188165, 62)
train target mean: 0.20944984956372323
valid target mean: 0.19897982420184906
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', 'ORIG_PRIOR_Driver', 'ORIG_PRIOR_Compound', 'ORIG_PRIOR_Race', 'ORIG_PRIOR_Stint', 'ORIG_PRIOR_PitStop', 'ORIG_PRIOR_Position', 'ORIG_PRIOR_LapNumber_bin10', 'ORIG_PRIOR_TyreLife_bin10', 'ORIG_PRIOR_RaceProgress_bin10', 'ORIG_PRIOR_Cumulative_Degradation_bin10', 'ORIG_PRIOR_LapTime_Delta_bin10', 'ORIG_PRIOR_Compound__Stint', 'ORIG_PRIOR_Compound__PitStop', 'ORIG_PRIOR_Race__Stint', 'ORIG_PRIOR_Race__PitStop', 'ORIG_PRIOR_Driver_minus_global', 'ORIG_PRIOR_Compound_minus_global', 'ORIG_PRIOR_Race_minus_global', 'ORIG_PRIOR_Stint_minus_global', 'O

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/4: val 1-auc_ovr = 0.053618
Epoch 2/4: val 1-auc_ovr = 0.048001
Epoch 3/4: val 1-auc_ovr = 0.047771
Epoch 4/4: val 1-auc_ovr = 0.045960


`Trainer.fit` stopped: `max_epochs=4` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 AUC: 0.95403954

CV RESULT
CV AUC: 0.9530389769769412
fold_scores: [0.9539405897464126, 0.9520531834151784, 0.95293943450267, 0.9522814383915083, 0.9540395396639483]
fold mean: 0.9530508371439435
fold std : 0.0008208398525796412


In [12]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

pd.DataFrame({
    CFG.ID_COL: train_id,
    CFG.TARGET: oof,
}).to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub_out = sub.copy()
target_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[target_col] = pred

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

final_feature_cols = feature_cols_by_fold[0]

feature_df = pd.DataFrame({
    "feature": final_feature_cols,
    "is_added_prior": [c in added_prior_features for c in final_feature_cols],
    "is_prior_minus_global": [c.startswith("ORIG_PRIOR_") and c.endswith("_minus_global") for c in final_feature_cols],
    "is_te": [c.endswith("TE") for c in final_feature_cols],
    "is_combo": [c in combo_names for c in final_feature_cols],
    "is_count_feature": [c.endswith("_count") for c in final_feature_cols],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

with open(CFG.OUTDIR / "prior_info.json", "w", encoding="utf-8") as f:
    json.dump(prior_info, f, ensure_ascii=False, indent=2)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "runtime": {
        "torch_version": torch.__version__,
        "pytabkit_version": version("pytabkit"),
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "realmlp_device": REALMLP_PARAMS.get("device"),
        "realmlp_n_ens": REALMLP_PARAMS.get("n_ens"),
        "realmlp_batch_size": REALMLP_PARAMS.get("batch_size"),
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_shape_raw": list(orig_full.shape),
        "original_path": str(CFG.ORIG_PATH),
        "id_col": CFG.ID_COL,
        "target": CFG.TARGET,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL_ROWS,
        "competition_target_mean": float(y.mean()),
        "original_target_mean": float(y_orig.mean()),
    },
    "features": {
        "base_cat_cols": base_cat_cols,
        "base_num_cols": base_num_cols,
        "new_cat_cols": new_cat_cols,
        "new_num_cols": new_num_cols,
        "combo_names": combo_names,
        "target_encoding": {
            "enabled": CFG.USE_TE,
            "te_cols": combo_names,
            "te_names": te_names,
            "note": "TargetEncoder is fit inside each outer fold on competition fold-train plus original fold-train rows.",
        },
        "orig_prior": {
            "enabled": True,
            "source": "original dataset target mean priors",
            "uses_year_prior": False,
            "uses_count_features": False,
            "uses_normalized_tyrelife": False,
            "added_prior_features": added_prior_features,
            "prior_info": prior_info,
        },
        "total_feature_count_before_te": len(feature_cols),
        "total_feature_count_after_te": len(final_feature_cols),
        "feature_columns": final_feature_cols,
        "notes": [
            "007 RealMLP shared001 min plus safe original prior graft.",
            "Normalized_TyreLife is dropped from original before any feature engineering.",
            "Original rows are appended only to each fold training split.",
            "ORIG_PRIOR_* mean and minus_global features are added.",
            "ORIG_PRIOR_Year and all *_count features are not added.",
            "Race_Compound and Race_Year interaction categories are target-encoded.",
            "OOF and pred are saved as npy for later correlation and blend.",
        ],
    },
    "model": {
        "family": "RealMLP",
        "library": "pytabkit",
        "estimator": "RealMLP_TD_Classifier",
        "params": REALMLP_PARAMS,
        "cv": {
            "scheme": "StratifiedKFold",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
    },
    "safety_checks": {
        "has_ORIG_PRIOR_Year": "ORIG_PRIOR_Year" in final_feature_cols,
        "has_ORIG_PRIOR_Year_count": "ORIG_PRIOR_Year_count" in final_feature_cols,
        "has_ORIG_PRIOR_Year_minus_global": "ORIG_PRIOR_Year_minus_global" in final_feature_cols,
        "count_feature_count": int(sum(c.endswith("_count") for c in final_feature_cols)),
        "count_features": [c for c in final_feature_cols if c.endswith("_count")],
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "prior_info": str(CFG.OUTDIR / "prior_info.json"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved artifacts to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(feature_df.head(120))
display(sub_out.head())


Saved artifacts to: /kaggle/working/exp_20260503_008_realmlp_shared001_plus_006b_orig_prior
Submission: /kaggle/working/exp_20260503_008_realmlp_shared001_plus_006b_orig_prior/submission_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.csv


,feature,is_added_prior,is_prior_minus_global,is_te,is_combo,is_count_feature
0,Driver,False,False,False,False,False
1,Compound,False,False,False,False,False
2,Race,False,False,False,False,False
3,Year,False,False,False,False,False
4,PitStop,False,False,False,False,False
...,...,...,...,...,...,...
57,ORIG_PRIOR_Compound__PitStop_minus_global,True,True,False,False,False
58,ORIG_PRIOR_Race__Stint_minus_global,True,True,False,False,False
59,ORIG_PRIOR_Race__PitStop_minus_global,True,True,False,False,False
60,_Race_Compound_TE,False,False,True,False,False


,id,PitNextLap
0,439140,0.004559
1,439141,0.017295
2,439142,0.005857
3,439143,0.228605
4,439144,0.781695
